# 🚀 Regelbasierte OCR-Nachbearbeitung

## Hinweise zur Ausführung des Notebooks
Dieses **Notebook** kann auf unterschiedlichen Levels erarbeitet werden (siehe Abschnitt ["Technische Voraussetzungen"](../introduction/introduction_requirements)): 
1. Book-Only Mode
2. Cloud Mode: Dafür auf 🚀 klicken und z.B. in Colab ausführen.
3. Local Mode: Dafür auf Herunterladen ↓ klicken und ".ipynb" wählen. 

## Übersicht
In diesem Notebook wird eine regelbasierte OCR-Nachkorrektur entwickelt und angewendet. Ziel ist es, typische Fehler, die beim OCR-Prozess historischer Texte entstehen, automatisch zu beheben und die Verbesserung messbar zu machen.

Dafür werden folgende Schritte durchgeführt:
1. Identifikation typischer OCR-Fehler im Korpus (z.B. `<` statt `ch`, `fie` statt `sie`, `ſ` statt `s`)
2. Implementierung von Korrekturregeln mit regulären Ausdrücken
3. Anwendung der Regeln auf ein Beispielbild mittels Tesseract-OCR
4. Messung der Verbesserung anhand von Precision, Recall und F1-Score im Vergleich mit einem Ground-Truth-Text
5. (Advanced) Anwendung der Korrekturregeln auf das gesamte Korpus

In [ ]:
!pip install pytesseract pillow Levenshtein

In [ ]:
import re
import pytesseract
from PIL import Image
from auxiliary.measure_ocr_quality import measure_ocr_quality

In [ ]:
# only for colab users, to download the auxiliary file with the function to measure the quality of the OCR output
import sys

if 'google.colab' in sys.modules:
    import os
    os.makedirs('auxiliary', exist_ok=True)
    !wget -q -O auxiliary/measure_ocr_quality.py https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/main/ocr_post_correction/auxiliary/measure_ocr_quality.py

## Typische Fehler

Im Folgenden listen wir einige typische Fehler in unserem Korpus auf:

* "fie" statt "sie" (Bild und Ergebnis später hinzufügen)
* "vm", "vnd" statt "um", "und"
* "<" statt "ch"

Einige Dinge sind keine Fehler, sondern Merkmale der historischen Orthographie, die wir für die weitere Verarbeitung mit modernen NLP-Tools normalisieren möchten:

* "ſ" statt "s"

In vielen Fällen können wir dies mit einigen regulären Such- und Ersetzungsmustern beheben (z.B. jedes `<`, das nicht von Leerzeichen umgeben ist, in `ch` umwandeln).

Der Standardweg, solche Muster auf einem Computer auszudrücken und zu implementieren, sind reguläre Ausdrücke. Mehr über reguläre Ausdrücke erfahren Sie [hier](https://www.w3schools.com/python/python_regex.asp).

<!-- In many cases we can fix it with some regular search-and replace patterns (e.g. take each `<` not surrounded by spaces and convert into `ch`)

The standard way to express & implement such patterns on a computer would be regular expressions. You can learn more about regular expressions [here](https://www.w3schools.com/python/python_regex.asp). -->

## Implementierung von Regeln für typische Fehler mit regulären Ausdrücken

In [ ]:
def post_correct_text(ocr_output):
    cleaner_output = re.sub(r'(\w)<(\w)', '\\1ch\\2', ocr_output)
    cleaner_output = re.sub(r'(\w)5(\w)', '\\1s\\2', cleaner_output)
    cleaner_output = re.sub(r'\bv(m|nd)\b', 'u\\1', cleaner_output)
    cleaner_output = re.sub(r'\bfie\b', 'sie', cleaner_output)
    cleaner_output = cleaner_output.replace('ſ','s')
    #cleaner_output = cleaner_output.replace('\n',' ')
    return cleaner_output

## Anwendung der Regeln auf die OCR-Ergebnisse <!-- ## Applying rules to OCR results --> 

In [ ]:
!wget https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/assets/images/grippe.jpeg

<img src="grippe.jpeg" width=700>

In [ ]:
ocr_output = pytesseract.image_to_string(Image.open('grippe.jpeg'), lang='frk')

In [ ]:
print(ocr_output)

In [ ]:
ocr_output_corr = post_correct_text(ocr_output)

Lass uns sehen, wie sich das Ganze verändert hat: <!-- Let us see how the whole thing changed: --> 

In [ ]:
print(ocr_output_corr)

## Messung der Verbesserung

Lass uns sehen, wie sich die regelbasierte Nachkorrektur auf die OCR-Qualitätsmetriken ausgewirkt hat

In [ ]:
ground_truth = """Die Grippe wütet weiter 
Zunahme der schweren Fälle in Berlin. 
Die Zahl der Grippefälle ist in den letzten 
beiden Tagen auch in Groß-Berlin noch 
erheblich gestiegen. Die Warenhäuser und son-
stigen großen Geschäfte, die Kriegs- und die pri-
vaten Betriebe klagen, daß übermäßig viele An-
gestellte sich haben krank melden müssen und auch 
bei der Post und bei der Straßenbahn ist der 
Prozentsatz der Grippekranken bedeutend ge-
stiegen."""

### Originales (unkorrigiertes) OCR-Ergebnis

In [ ]:
precision, recall, f_score = measure_ocr_quality(ocr_output, ground_truth)

In [ ]:
print(f'Precision: {round(precision, 4)}\nRecall: {round(recall, 4)}\nF1-score: {round(f_score, 4)}')

### Korrigiertes OCR-Ergebnis

In [ ]:
precision, recall, f_score = measure_ocr_quality(ocr_output_corr, ground_truth)

In [ ]:
print(f'Precision: {round(precision, 4)}\nRecall: {round(recall, 4)}\nF1-score: {round(f_score, 4)}')

Also, unsere F-Score hat sich etwas verbessert, gut! 

## (Advanced) Ausführung des regelbasierten OCR-Nachkorrekturverfahrens auf dem gesamten Korpus

In [ ]:
from pathlib import Path
from tqdm import tqdm

In [ ]:
pathtxt = Path('../data/txt')
if not pathtxt.exists():
    pathtxt.mkdir(parents=True)

In [ ]:
for file in tqdm(pathtxt.iterdir()):
    if file.suffix == '.txt':
        text = file.read_text()
        corrected = post_correct_text(text)
        file.write_text(corrected)